# ArbOS 60 : step-by-step walkthrough

Method-by-method execution of the ArbOS 60 dynamic-pricing math, with each
function inlined into its own cell (no `Arbos60GasPricing` class import).
Runs on a fixed 1,000-tx slice from `data/multigas_usage_extracts/2026-01/per_tx.parquet`.

Companion: `docs/arbos51_vs_arbos60_equations.md`.

## 0. Setup : load 1,000 priced txs

Filter: drop sub-30 K-gas txs (degenerate corner, e.g. value-only ETH transfers
or 0-gas internal txs). Take the first 1,000 priced txs from the Jan parquet —
they sit inside a ~30-second window, enough to see one-second-tick backlog
dynamics on this small slice.

In [1]:
import pathlib, sys
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Walk up until we find data/multigas_usage_extracts so the notebook works
# whether the kernel started in notebooks/ or at the repo root.
ROOT = pathlib.Path().resolve()
while not (ROOT / 'data' / 'multigas_usage_extracts').exists():
    if ROOT == ROOT.parent:
        raise SystemExit('repo root (with data/multigas_usage_extracts/) not found')
    ROOT = ROOT.parent

MIN_GAS   = 30_000
TX_PQ     = ROOT / 'data' / 'multigas_usage_extracts' / '2026-01' / 'per_tx.parquet'
BLOCKS_PQ = ROOT / 'data' / 'multigas_usage_extracts' / '2026-01' / 'blocks.parquet'

tx = (
    pl.scan_parquet(str(TX_PQ))
      .filter((pl.col('computation') + pl.col('wasmComputation')
               + pl.col('storageAccess') + pl.col('storageGrowth')
               + pl.col('historyGrowth') + pl.col('l2Calldata')
               + pl.col('l1Calldata')) >= MIN_GAS)
      .head(1000)
      .collect()
)

# Per-block resource sums — input shape per_block_resource_gas wants.
blocks_agg = (
    tx.group_by('block').agg([
        pl.col('computation').sum(),
        pl.col('wasmComputation').sum(),
        pl.col('storageAccessRead').sum(),
        pl.col('storageAccessWrite').sum(),
        pl.col('storageGrowth').sum(),
        pl.col('historyGrowth').sum(),
        pl.col('l2Calldata').sum(),
    ]).sort('block').rename({'block': 'block_number'})
)

# Block timestamps from the blocks parquet (filter to our slice).
block_times = (
    pl.scan_parquet(str(BLOCKS_PQ))
      .filter(pl.col('block_num').is_in(blocks_agg['block_number'].to_list()))
      .select(pl.col('block_num').alias('block_number'),
              pl.col('block_time').str.to_datetime(format='%Y-%m-%dT%H:%M:%S%z')
                  .dt.replace_time_zone(None).alias('block_time'))
      .collect()
)
blocks = blocks_agg.join(block_times, on='block_number', how='inner').sort('block_number')

print(f'txs:    {tx.height:>6,}   blocks: {blocks.height:>4}')
print(f"window: {blocks['block_time'].min()} → {blocks['block_time'].max()}")
blocks.head()

txs:     1,000   blocks:  119
window: 2026-01-01 00:00:00 → 2026-01-01 00:00:29


block_number,computation,wasmComputation,storageAccessRead,storageAccessWrite,storageGrowth,historyGrowth,l2Calldata,block_time
i64,i64,i64,i64,i64,i64,i64,i64,datetime[μs]
416593974,3581986,0,1322800,121800,0,37120,37000,2026-01-01 00:00:00
416593975,360569,0,455500,5800,0,0,32340,2026-01-01 00:00:00
416593976,478236,0,363600,11600,60000,9216,46240,2026-01-01 00:00:00
416593977,507528,0,380500,40600,30000,9984,53748,2026-01-01 00:00:00
416593978,377295,0,334100,29000,80000,11776,40424,2026-01-01 00:00:01


## 1. Constants + constraint sets

$p_{\min} = 0.02$ gwei. Four weighted-inequality sets:

$$ \sum_k a_{i,k}\, g_k \le T_{i,j}, \qquad i \in \{1,2,3,4\},\; j \in \text{set}_i\text{'s ladder}. $$

Sets 1–3 each have a 6-rung geometric ladder; set 4 has one rung.

In [2]:
# Minimum L2 base fee (gwei) — same floor for every priced resource.
P_MIN_GWEI = 0.02

# Six priced resources. Order = dimension index k for p_k.
PRICED_SYMBOLS = ['c', 'sw', 'sr', 'sg', 'hg', 'l2']
PRICED_SYMBOL_LABELS = {
    'c':  'Computation',
    'sw': 'Storage Write',
    'sr': 'Storage Read',
    'sg': 'Storage Growth',
    'hg': 'History Growth',
    'l2': 'L2 Calldata',
}

# Per-set weights a_{i,k} on the priced resources.
SET_WEIGHTS = {
    'First Set — Storage/Compute mix 1':   {'c': 1.0,    'sw': 0.67, 'sr': 0.14, 'sg': 0.06},
    'Second Set — Storage/Compute mix 2':  {'c': 0.0625, 'sw': 1.0,  'sr': 0.21, 'sg': 0.09},
    'Third Set — History Growth':           {'hg': 1.0},
    'Fourth Set — Long-term Disk Growth':   {'sw': 0.8812, 'sg': 0.2526, 'hg': 0.301, 'l2': 1.0},
}

# Per-set ladders: list of (T Mgas/s, A seconds).
SET_LADDERS = {
    'First Set — Storage/Compute mix 1': [
        (15.40, 10_000), (20.41, 2_861), (27.06,   819),
        (35.86,    234), (47.53,    67), (63.00,    19),
    ],
    'Second Set — Storage/Compute mix 2': [
        ( 3.13, 10_000), ( 4.16, 4_488), ( 5.53, 2_014),
        ( 7.35,    904), ( 9.77,   406), (12.99,   182),
    ],
    'Third Set — History Growth': [
        ( 67.30, 10_000), ( 81.27, 1_591), ( 98.14,   253),
        (118.50,     40), (143.10,     6), (172.80,     1),
    ],
    'Fourth Set — Long-term Disk Growth': [
        (2.30, 36_000),
    ],
}

# Per-set colour palette — used by every later cell.
set_colors = {n: c for n, c in zip(SET_LADDERS.keys(),
              ['#0a2d6e', '#b54a00', '#0f5f0f', '#7a1b1b'])}

print(f'p_min = {P_MIN_GWEI} gwei\n')

# ── Weights table ─────────────────────────────────────────────────
print('SET_WEIGHTS  (a_{i,k})')
header = f'  {"set":<40s}  ' + '  '.join(f'{s:>6}' for s in PRICED_SYMBOLS)
print(header)
print('  ' + '-' * (len(header) - 2))
for name, w in SET_WEIGHTS.items():
    row = f'  {name.split(" — ")[0]:<40s}  ' + '  '.join(
        f'{w.get(s, 0.0):>6.4g}' for s in PRICED_SYMBOLS)
    print(row)

# ── Ladders table — one block per set ─────────────────────────────
print('\nSET_LADDERS  (T Mgas/s, A seconds)')
for name, ladder in SET_LADDERS.items():
    print(f'\n  {name}')
    print(f'  {"j":>2}  {"T":>9}  {"A":>9}')
    print('  ' + '-' * 25)
    for j, (T, A) in enumerate(ladder):
        print(f'  {j:>2d}  {T:>9.2f}  {A:>9,d}')


p_min = 0.02 gwei

SET_WEIGHTS  (a_{i,k})
  set                                            c      sw      sr      sg      hg      l2
  ----------------------------------------------------------------------------------------
  First Set                                      1    0.67    0.14    0.06       0       0
  Second Set                                0.0625       1    0.21    0.09       0       0
  Third Set                                      0       0       0       0       1       0
  Fourth Set                                     0  0.8812       0  0.2526   0.301       1

SET_LADDERS  (T Mgas/s, A seconds)

  First Set — Storage/Compute mix 1
   j          T          A
  -------------------------
   0      15.40     10,000
   1      20.41      2,861
   2      27.06        819
   3      35.86        234
   4      47.53         67
   5      63.00         19

  Second Set — Storage/Compute mix 2
   j          T          A
  -------------------------
   0       3.13     10,000
  

## 2. `block_seconds_utc` : block timestamps to integer UTC seconds

$$ s_b = \lfloor t_b \rfloor_{\text{1 s}} \quad (\text{int64 unix seconds}). $$

Multiple blocks land in the same `s` (Arbitrum target ≈ 250 ms blocks → ~4 blocks/s).

In [3]:
def block_seconds_utc(block_times_utc):
    """Per-block integer-second UTC timestamps."""
    return block_times_utc.astype('datetime64[s]').astype(np.int64)

# `blocks` is already sorted by block_number in setup; block_number is
# monotone in time, so block_seconds is monotone non-decreasing.
block_seconds = block_seconds_utc(blocks['block_time'].to_numpy())

n_seconds = len(np.unique(block_seconds))
print(f'blocks          : {blocks.height}')
print(f'unique seconds  : {n_seconds}')
print(f'avg blocks / s  : {blocks.height / n_seconds:.2f}')
# Show the first 10 entries — block_number side-by-side with its second.
print('first 10 (block_number → second):')
for b, s in zip(blocks['block_number'][:10].to_list(), block_seconds[:10]):
    print(f'  {b:>10,d}  →  {s}')


blocks          : 119
unique seconds  : 30
avg blocks / s  : 3.97
first 10 (block_number → second):
  416,593,974  →  1767225600
  416,593,975  →  1767225600
  416,593,976  →  1767225600
  416,593,977  →  1767225600
  416,593,978  →  1767225601
  416,593,979  →  1767225601
  416,593,980  →  1767225601
  416,593,981  →  1767225601
  416,593,982  →  1767225602
  416,593,983  →  1767225602


## 3. `aggregate_per_second` : per-block values into per-second buckets

$$ \text{inflow}(s) = \sum_{b: s_b = s} g_{\text{total}}(b). $$

Empty seconds keep contributing `0` to inflow, so the recursion still drains
them. Implemented with `np.bincount` for vector speed; assumes
`block_seconds` is sorted ascending.

In [4]:
def aggregate_per_second(values_per_block, block_seconds):
    """Sum per-block values into per-second buckets, dense from min..max
    of block_seconds. Empty seconds (no blocks landed) contribute 0."""
    s_min     = int(block_seconds[0])
    s_max     = int(block_seconds[-1])
    n_seconds = s_max - s_min + 1
    seconds   = np.arange(s_min, s_max + 1, dtype=np.int64)
    idx       = (block_seconds - s_min).astype(np.int64)
    summed    = np.bincount(
        idx,
        weights=np.asarray(values_per_block, dtype=np.float64),
        minlength=n_seconds,
    )
    return seconds, summed


total_gas_pb = (blocks['computation'] + blocks['wasmComputation']
                + blocks['storageAccessRead'] + blocks['storageAccessWrite']
                + blocks['storageGrowth'] + blocks['historyGrowth']
                + blocks['l2Calldata']).to_numpy().astype(np.float64)

seconds_axis, inflow_per_sec = aggregate_per_second(total_gas_pb, block_seconds)

fig = go.Figure(go.Bar(
    x=seconds_axis - seconds_axis[0],
    y=inflow_per_sec / 1e6,
    marker_color='#1f77b4', name='inflow',
    hovertemplate='s+%{x}: %{y:.2f} Mgas/s<extra></extra>',
))
fig.update_layout(template='plotly_white', height=380,
                  title='Per-second total-gas inflow over the 1,000-tx window',
                  xaxis_title='seconds since window start',
                  yaxis_title='inflow (Mgas / s)')
fig.show()
print(f'window length: {len(seconds_axis)} s, mean inflow: {inflow_per_sec.mean()/1e6:.2f} Mgas/s')

window length: 30 s, mean inflow: 7.58 Mgas/s


## 4. `taylor4_exp` : degree-4 Taylor approximation of `exp`

$$ \text{taylor4}(x) = 1 + x + \tfrac{x^2}{2} + \tfrac{x^3}{6} + \tfrac{x^4}{24}. $$

Matches Nitro's `ApproxExpBasisPoints(x, 4)`

In [5]:
def taylor4_exp(x):
    """Degree-4 Taylor approximation of exp(x)."""
    x2 = x * x
    x3 = x2 * x
    x4 = x2 * x2
    return 1.0 + x + x2 / 2.0 + x3 / 6.0 + x4 / 24.0


x = np.linspace(0, 5, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=np.exp(x),     mode='lines', name='np.exp',  line=dict(color='#000', width=2)))
fig.add_trace(go.Scatter(x=x, y=taylor4_exp(x), mode='lines', name='taylor4', line=dict(color='#1f77b4', dash='dash')))
fig.update_layout(template='plotly_white', height=360,
                  title='exp vs taylor4', xaxis_title='x', yaxis_title='value', yaxis_type='log')
fig.show()

## 5. `per_block_resource_gas` : per-block 6-Vector

Six priced symbols `(c, sw, sr, sg, hg, l2)` aggregated from the per-tx
multigas columns. `c = computation + wasmComputation`; the rest map 1:1
from the columns we summed in setup.

In [6]:
def per_block_resource_gas(blocks):
    """Per-block resource gas (raw gas units), keyed by the 6 priced symbols."""
    return {
        'c':  (blocks['computation'].to_numpy()
               + blocks['wasmComputation'].to_numpy()).astype(np.float64),
        'sw': blocks['storageAccessWrite'].to_numpy().astype(np.float64),
        'sr': blocks['storageAccessRead'].to_numpy().astype(np.float64),
        'sg': blocks['storageGrowth'].to_numpy().astype(np.float64),
        'hg': blocks['historyGrowth'].to_numpy().astype(np.float64),
        'l2': blocks['l2Calldata'].to_numpy().astype(np.float64),
    }


parts = per_block_resource_gas(blocks)
for k, v in parts.items():
    print(f'  {k:>4}  total={v.sum()/1e6:>9.2f} Mgas   max-block={v.max()/1e3:>7.1f} Kgas')

# Stacked bar of resource composition per block (in Mgas).
colors = {'c':'#1f77b4','sw':'#2ca02c','sr':'#98df8a','sg':'#d62728','hg':'#ff7f0e','l2':'#9467bd'}
fig = go.Figure()
for k in PRICED_SYMBOLS:
    fig.add_trace(go.Bar(name=PRICED_SYMBOL_LABELS[k],
                         x=blocks['block_number'].to_numpy(),
                         y=parts[k] / 1e6, marker_color=colors[k]))
fig.update_layout(template='plotly_white', barmode='stack', height=380,
                  title='Per-block resource gas (stacked)',
                  xaxis_title='block', yaxis_title='Mgas')
fig.show()

     c  total=    91.73 Mgas   max-block= 9204.7 Kgas
    sw  total=     9.17 Mgas   max-block=  884.5 Kgas
    sr  total=    67.83 Mgas   max-block= 7297.8 Kgas
    sg  total=    47.54 Mgas   max-block=12400.0 Kgas
    hg  total=     4.12 Mgas   max-block=  624.6 Kgas
    l2  total=     7.08 Mgas   max-block=  147.2 Kgas


## 6. `backlog_per_second` — the 1-second tick recursion

$$ B(s+1) = \max\!\left(0,\; B(s) + \text{inflow}(s) - T \cdot 1 \right). $$

Implementation: a plain Python loop over wall-clock seconds; within each
iteration `np.maximum` updates `B` for every rung of the ladder at once
(vectorised across the T-axis).

We record `out[:, s] = B` **before** the second-`s` update, so the array
holds the *start-of-second* backlog — i.e. the `B` value a block landing
in second `s` reads, before that second's inflow / drain happen. ~4
blocks land per second, so they all see the same `B[s]`.


In [ ]:
def backlog_per_second(inflow_per_second, T_arr):
    """Start-of-each-second backlog seen by blocks landing in that second.

        B(0) = 0;   B(s+1) = max(0, B(s) + inflow(s) - T*1).

    `T_arr` is the 1-D ladder (Mgas/s); returns shape (n_T, n_sec).
    out[:, s] is recorded *before* second s's update — that's exactly
    what a block in second s reads.
    """
    drain = T_arr * 1e6                                       # Mgas/s -> gas/s
    out = np.empty((T_arr.shape[0], inflow_per_second.shape[0]))
    B   = np.zeros(T_arr.shape[0])
    for s, inflow_s in enumerate(inflow_per_second):
        out[:, s] = B                                         # snapshot before update
        B = np.maximum(0.0, B + inflow_s - drain)             # advance to B(s+1)
    return out

# Plot
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[s.split(' — ')[0] for s in SET_WEIGHTS.keys()],
                    shared_xaxes=True, vertical_spacing=0.10)

for idx, (set_name, weights) in enumerate(SET_WEIGHTS.items()):
    r, c = idx // 2 + 1, idx % 2 + 1
    inflow_pb               = sum(w * parts[sym] for sym, w in weights.items())
    seconds_axis, inflow_ps = aggregate_per_second(inflow_pb, block_seconds)

    ladder = SET_LADDERS[set_name]
    T_arr  = np.array([t for t, _ in ladder], dtype=np.float64)
    B      = backlog_per_second(inflow_ps, T_arr)              # (n_T, n_sec)

    for j, ((T, A), Brow) in enumerate(zip(ladder, B)):
        fig.add_trace(go.Scatter(
            x=seconds_axis - seconds_axis[0], y=Brow / 1e6,
            mode='lines',
            name=f'{set_name.split(" — ")[0]} C{j} (T={T}, A={A}s)',
            line=dict(color=set_colors[set_name]),
            showlegend=False,
        ), row=r, col=c)

fig.update_layout(template='plotly_white', height=560,
                  title='Backlog B<sub>i,j</sub>(s) — every (set, constraint) pair')
for ax in (1, 2, 3, 4):
    fig.update_yaxes(title_text='backlog (Mgas)', row=(ax-1)//2 + 1, col=(ax-1)%2 + 1)
    fig.update_xaxes(title_text='seconds since window start', row=(ax-1)//2 + 1, col=(ax-1)%2 + 1)
fig.show()


## 7. `compute_set_exponents` : per-set raw exponent

$$ E_i(t) = \sum_j \frac{B_{i,j}(t)}{A_{i,j}\,T_{i,j}}. $$

One number per set per second. The four `E_i` curves are what the
per-resource max-over-sets projects on top of.

In [ ]:
def compute_set_exponents(parts, block_seconds, set_weights, set_ladders):
    """E_i(s) = Σ_j B_{i,j}(s) / (A_{i,j}*T_{i,j})   — per-second, per set."""
    E_per_set   = {}
    seconds_ref = None
    for set_name, weights in set_weights.items():
        inflow_pb               = sum(w * parts[sym] for sym, w in weights.items())
        seconds_ref, inflow_ps  = aggregate_per_second(inflow_pb, block_seconds)
        ladder = set_ladders[set_name]
        T_arr  = np.array([t for t, _ in ladder], dtype=np.float64)
        A_arr  = np.array([a for _, a in ladder], dtype=np.float64)
        B      = backlog_per_second(inflow_ps, T_arr)                  # (n_T, n_sec)
        E_per_set[set_name] = (B / (A_arr * T_arr * 1e6)[:, None]).sum(axis=0)
    return seconds_ref, E_per_set


seconds_axis, E_per_set = compute_set_exponents(parts, block_seconds, SET_WEIGHTS, SET_LADDERS)

# Plot
fig = go.Figure()
for s, E in E_per_set.items():
    fig.add_trace(go.Scatter(x=seconds_axis - seconds_axis[0], y=E,
                             mode='lines', name=s.split(' — ')[0],
                             line=dict(color=set_colors[s])))
fig.update_layout(template='plotly_white', height=380,
                  title='Per-set raw exponent E<sub>i</sub>(s)',
                  xaxis_title='seconds since window start', yaxis_title='E<sub>i</sub>')
fig.show()
print({s.split(' — ')[0]: float(E.max()) for s, E in E_per_set.items()})


{'First Set': 0.0, 'Second Set': 2.229054912140575e-05, 'Third Set': 0.0, 'Fourth Set': 1.847075797101449e-05}


## 8. `price_per_resource` : max-over-sets projection

$$ p_k = p_{\min}\,\exp\!\left(\max_i\,\{a_{i,k}\,E_i\}\right). $$

Only the **binding** set lifts $p_k$. Sets that give resource $k$ a zero
weight ($a_{i,k}=0$) drop out of the max for that resource. With $e_k \ge 0$
and `taylor4_exp` monotone increasing on $[0, \infty)$, the floor
$p_k \ge p_{\min}$ is preserved.

In [ ]:
def price_per_resource(E_per_set, set_weights, p_min_gwei):
    """p_k(s) = p_min * taylor4_exp(max_i a_{i,k} * E_i(s)).  Per-second."""
    prices = {}
    n_sec  = next(iter(E_per_set.values())).shape[0]
    for k in PRICED_SYMBOLS:
        e_k = np.zeros(n_sec)
        for set_name, weights in set_weights.items():
            a_ik = float(weights.get(k, 0.0))
            if a_ik == 0.0:
                continue
            e_k = np.maximum(e_k, a_ik * E_per_set[set_name])
        prices[k] = p_min_gwei * taylor4_exp(np.maximum(e_k, 0.0))
    return prices


prices = price_per_resource(E_per_set, SET_WEIGHTS, P_MIN_GWEI)

# Plot
fig = go.Figure()
for k in PRICED_SYMBOLS:
    fig.add_trace(go.Scatter(
        x=seconds_axis - seconds_axis[0], y=prices[k],
        mode='lines', name=f'p_{k} — {PRICED_SYMBOL_LABELS[k]}',
        line=dict(color=colors[k]),
    ))
fig.add_hline(y=P_MIN_GWEI, line_dash='dot', line_color='rgba(0,0,0,0.4)',
              annotation_text=f'p_min = {P_MIN_GWEI} gwei')
fig.update_layout(template='plotly_white', height=400,
                  title='Per-resource price p<sub>k</sub>(s)',
                  xaxis_title='seconds since window start', yaxis_title='price (gwei)')
fig.show()
print({k: f'{v.min():.4f} → {v.max():.4f}' for k, v in prices.items()})


{'c': '0.0200 → 0.0200', 'sw': '0.0200 → 0.0200', 'sr': '0.0200 → 0.0200', 'sg': '0.0200 → 0.0200', 'hg': '0.0200 → 0.0200', 'l2': '0.0200 → 0.0200'}


## 9. `per_tx_resource_split` + `fee_per_tx` : per transaction pricing

$$ \text{fee}_{tx} = \sum_k g_{tx,k}\, p_k(\text{second of } tx) \quad [\text{ETH}]. $$

Dimensionally a fee, not a per-gas price.

In [ ]:
def per_tx_resource_split(computation, wasm_computation, storage_access_read,
                          storage_access_write, storage_growth, history_growth,
                          l2_calldata):
    """Per-tx 6-resource gas dict from raw multigas columns. l1Calldata
    is intentionally absent — ArbOS 60 prices it via the L1 fee path."""
    return {
        'c':  computation + wasm_computation,
        'sw': storage_access_write,
        'sr': storage_access_read,
        'sg': storage_growth,
        'hg': history_growth,
        'l2': l2_calldata,
    }


def fee_per_tx(prices_per_sec, tx_sec_idx, tx_g_per_resource):
    """fee_tx = Σ_k g_tx,k * p_k(second of tx).  Gwei (= ETH * 1e9)."""
    fee = np.zeros(tx_sec_idx.shape[0])
    for k in PRICED_SYMBOLS:
        fee = fee + tx_g_per_resource[k] * prices_per_sec[k][tx_sec_idx]
    return fee


# Map each tx -> the index of its block's wall-clock second in seconds_axis.
block_to_sec_idx = (block_seconds - seconds_axis[0]).astype(np.int64)
block_to_idx     = {int(b): i for i, b in enumerate(blocks['block_number'].to_numpy())}
tx_block_idx     = np.array([block_to_idx[int(b)] for b in tx['block'].to_numpy()], dtype=np.int64)
tx_sec_idx       = block_to_sec_idx[tx_block_idx]

def _np(col):
    return tx[col].cast(pl.Float64).to_numpy()

tx_g = per_tx_resource_split(
    computation          = _np('computation') + _np('wasmComputation'),
    wasm_computation     = np.zeros(tx.height),  # already folded into the c term
    storage_access_read  = _np('storageAccessRead'),
    storage_access_write = _np('storageAccessWrite'),
    storage_growth       = _np('storageGrowth'),
    history_growth       = _np('historyGrowth'),
    l2_calldata          = _np('l2Calldata'),
)
fee_tx = fee_per_tx(prices, tx_sec_idx, tx_g)

# Bucket per-tx fees back to per second for the time series.
n_sec       = seconds_axis.shape[0]
fee_per_sec = np.bincount(tx_sec_idx, weights=fee_tx, minlength=n_sec)

# Plot
fig = go.Figure(go.Scatter(
    x=seconds_axis - seconds_axis[0], y=fee_per_sec / 1e9,    # gwei -> ETH
    mode='lines', line=dict(color='#1f77b4'),
))
fig.update_layout(template='plotly_white', height=380,
                  title='Σ fee_tx per second  (ETH)',
                  xaxis_title='seconds since window start',
                  yaxis_title='fee (ETH)')
fig.show()
print(f'mean fee   = {fee_tx.mean() / 1e9:>16.6f} ETH')
print(f'total fee  = {fee_tx.sum() / 1e9:>16.6f} ETH')


mean fee   =         0.000005 ETH
total fee  =         0.004550 ETH
